# Build scenario-level dataset for scenario discovery

Reads the Scenario Compass Initiative pathways ensemble (`data/SCI-2025_v1.0_pathways_ensemble_global.xlsx`) and produces one clean, scenario-level table (one row per Model-Scenario) with:

- the `meta` sheet indicators (climate category, year of net-zero CO2, vetting flags, ...)
- a set of pathway-descriptor variables pivoted out of the long-format `data` sheet (CO2 emissions, primary/final energy, electricity, carbon removal - full 2010-2100 trajectories)
- the net-zero-by-2070 outcome labels (`success_nz2070`, `failure_nz2070`, `sustained_success_nz2070`)

Output: `script/data_for_scenariodiscovery_full.csv`, used as the input to the PRIM/CART notebook.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)

In [2]:
XLSX_PATH = "../data/SCI-2025_v1.0_pathways_ensemble_global.xlsx"
ENGINE = "calamine"  # python-calamine: much faster than openpyxl on this 120MB file

OUTPUT_CSV = "data_for_scenariodiscovery_full.csv"

## 1. Load the `meta` sheet

One row per Model-Scenario. Contains climate diagnostics, vetting flags, and - importantly -
`Emissions Diagnostics|Year of Net Zero|CO2`, which we use directly to build the net-zero outcome
(no need to derive it from the raw CO2 trajectory).

In [3]:
meta = pd.read_excel(XLSX_PATH, sheet_name="meta", engine=ENGINE)
print(meta.shape)
meta.head()

(1599, 71)


,Model,Scenario,Climate Assessment|Exceedance Probability 1.5°C [MAGICCv7.5.3],Climate Assessment|Exceedance Probability 2.0°C [MAGICCv7.5.3],Climate Assessment|Exceedance Probability 2.5°C [MAGICCv7.5.3],Climate Assessment|Exceedance Probability 3.0°C [MAGICCv7.5.3],Climate Assessment|Peak Warming|33rd Percentile [MAGICCv7.5.3],Climate Assessment|Peak Warming|67th Percentile [MAGICCv7.5.3],Climate Assessment|Peak Warming|Median [MAGICCv7.5.3],Climate Assessment|Warming in 2100|33rd Percentile [MAGICCv7.5.3],Climate Assessment|Warming in 2100|67th Percentile [MAGICCv7.5.3],Climate Assessment|Warming in 2100|Median [MAGICCv7.5.3],Climate Assessment|Year of Peak Warming|33rd Percentile [MAGICCv7.5.3],Climate Assessment|Year of Peak Warming|67th Percentile [MAGICCv7.5.3],Climate Assessment|Year of Peak Warming|Median [MAGICCv7.5.3],Climate Category|AR6 [ID],Climate Category|AR6 [Name],Climate Category|SCI 2025 [Tier III],Climate Category|SCI 2025 [Tier II],Climate Category|SCI 2025 [Tier I],Data Source (DOI),"Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]","Emissions Diagnostics|Cumulative CO2 [2020-2100, Gt CO2]","Emissions Diagnostics|Cumulative Kyoto Gases [2020-2100, Gt CO2e]","Emissions Diagnostics|Cumulative Net-Negative CO2 [2020-2100, Gt CO2]",Emissions Diagnostics|Year of Net Zero|CO2,Emissions Diagnostics|Year of Net Zero|Kyoto Gases,Feasibility Concern|Carbon Capture|World|2030,Feasibility Concern|Carbon Capture|World|2035,Feasibility Concern|Carbon Capture|World|2040,...,Historical Vetting|Final Energy|2020,Historical Vetting|Final Energy|2025,Historical Vetting|Primary Energy|Coal|2010,Historical Vetting|Primary Energy|Coal|2015,Historical Vetting|Primary Energy|Coal|2020,Historical Vetting|Primary Energy|Coal|2025,Historical Vetting|Primary Energy|Gas|2010,Historical Vetting|Primary Energy|Gas|2015,Historical Vetting|Primary Energy|Gas|2020,Historical Vetting|Primary Energy|Gas|2025,Historical Vetting|Primary Energy|Nuclear|2010,Historical Vetting|Primary Energy|Nuclear|2015,Historical Vetting|Primary Energy|Nuclear|2020,Historical Vetting|Primary Energy|Nuclear|2025,Historical Vetting|Primary Energy|Oil|2010,Historical Vetting|Primary Energy|Oil|2015,Historical Vetting|Primary Energy|Oil|2020,Historical Vetting|Primary Energy|Oil|2025,Legacy Mapping|AR6|Model Name,Legacy Mapping|AR6|Scenario Name,Project,Scenario Ensemble|AR6,Scenario Ensemble|SSP,Scientific Manuscript (Citation),Scientific Manuscript (DOI),Sustainability Concern|Exceeding Prudent Limit For Geological Carbon Storage|World,Sustainability Concern|Food Availability|World,Sustainability Concern|Unsustainable Bioenergy Use|World,Vetting|SCI 2025,version
0,AIM/CGE 2.0,SSP1-19,53.166667,6.500000,0.166667,0.000000,1.411097,1.628765,1.518461,1.001548,1.253812,1.132453,2037.0,2039.0,2038.0,C1,C1: Below 1.5°C with no or limited overshoot,GW1a,GW1,GW1,NaN,348.340310,324.319294,665.101646,-166.408314,2055.0,2066.0,high,medium,ok,...,ok,ok,ok,ok,ok,error,ok,ok,ok,error,ok,error,ok,error,ok,ok,ok,ok,NaN,NaN,SSP,NaN,1.0,Rogelj et al. (2018),10.1038/s41558-018-0091-3,ok,ok,ok,failed,1
1,AIM/CGE 2.0,SSP1-26,74.500000,21.333333,5.666667,0.666667,1.561912,1.844314,1.702569,1.443959,1.789941,1.609574,2049.0,2069.0,2059.0,C3,C3: Likely below 2°C,GW2-IIIb,GW2b,GW2,NaN,455.449049,988.478191,1501.479227,NaN,NaN,NaN,medium,ok,ok,...,ok,ok,ok,ok,ok,error,ok,ok,ok,error,ok,error,ok,error,ok,ok,ok,ok,AIM/CGE 2.0,SSP1-26,SSP,1.0,1.0,Riahi et al. (2017),10.1016/j.gloenvcha.2016.05.009,ok,ok,ok,failed,1
2,AIM/CGE 2.0,SSP1-34,93.500000,57.666667,22.166667,8.833333,1.908544,2.320240,2.098823,1.900225,2.320240,2.090244,2094.0,2100.0,2100.0,C5,C5: Below 2.5°C,GW5b,GW5,GW5,NaN,159.484812,1808.796903,2467.636562,NaN,NaN,NaN,medium,ok,ok,...,ok,ok,ok,ok,ok,error,ok,ok,ok,error,ok,error,ok,error,ok,ok,ok,ok,AIM/CGE 2.0,SSP1-34,SSP,1.0,1.0,Riahi et al. (2017),10.1016/j.gloenvcha.2016.05.009,ok,ok,ok,failed,1
3,AIM/CGE 2.0,SSP1-45,99.833333,90.000000,61.166667,28.000000,2.419310,

In [4]:
# Sanity check on the field the whole outcome label depends on
meta["Emissions Diagnostics|Year of Net Zero|CO2"].describe()

count     909.000000
mean     2070.380638
std        14.398448
min      2030.000000
25%      2060.000000
50%      2070.000000
75%      2080.000000
max      2100.000000
Name: Emissions Diagnostics|Year of Net Zero|CO2, dtype: float64

In [5]:
# Worth flagging: most rows are marked 'failed' on SCI 2025 vetting. This is a multi-model ensemble
# and 'failed' here means a historical-vetting check tripped on at least one variable, not that the
# scenario is unusable. Keep the flag in the dataset rather than silently dropping these rows -
# decide deliberately later whether to filter on it.
meta["Vetting|SCI 2025"].value_counts(dropna=False)

Vetting|SCI 2025
failed                    1237
ok                         347
insufficient reporting      11
NaN                          4
Name: count, dtype: int64

## 2. Load the `data` sheet (long IAMC format) and pivot to wide

548k rows x (Model, Scenario, Region, Variable, Unit, year columns). Region is always `World` in this
file (the five-region breakdown lives in a separate file you mentioned providing later).

We keep full 2010-2100 trajectories (not just single-year snapshots) for a small set of pathway
variables, so we can both plot trajectories and derive extra features (decline rates, sustained
net-zero) later without re-reading the 120MB file.

In [6]:
data_long = pd.read_excel(XLSX_PATH, sheet_name="data", engine=ENGINE)
print(data_long.shape)
assert data_long["Region"].unique().tolist() == ["World"]
data_long.head()

(548541, 24)


,Model,Scenario,Region,Variable,Unit,2010,2015,2020,2025,2030,2035,2040,2045,2050,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
0,AIM/CGE 2.0,SSP1-19,World,Agricultural Demand|Crops,million t DM/yr,3489.8126,3711.06720,3932.3218,4119.09705,4305.8723,4464.12810,4622.3839,4732.95650,4843.5291,NaN,4945.6105,NaN,4961.4156,NaN,4876.1981,NaN,4703.0300,NaN,4439.7738
1,AIM/CGE 2.0,SSP1-19,World,Agricultural Demand|Livestock,million t DM/yr,195.2943,206.86445,218.4346,228.28975,238.1449,246.54265,254.9404,260.69025,266.4401,NaN,271.9541,NaN,273.0258,NaN,269.1877,NaN,260.7863,NaN,248.0497
2,AIM/CGE 2.0,SSP1-19,World,Agricultural Production|Crops|Energy Crops,million t DM/yr,0.0000,10.23115,20.4623,105.94275,191.4232,634.51415,1077.6051,1268.07845,1458.5518,NaN,2211.0671,NaN,2638.0130,NaN,2722.9841,NaN,2828.1186,NaN,2890.7464
3,AIM/CGE 2.0,SSP1-19,World,Capacity|Electricity,GW,4164.7229,4871.75230,5578.7817,7885.12530,10191.4689,13569.41000,16947.3511,19729.05220,22510.7533,NaN,26292.3429,NaN,27791.9200,NaN,28294.8670,NaN,27904.1215,NaN,27345.7683
4,AIM/CGE 2.0,SSP1-19,World,Capacity|Electricity|Biomass,GW,61.5660,65.00060,68.4352,89.68315,110.9311,244.40660,377.8821,427.99240,478.1027,NaN,716.4235,NaN,868.6942,NaN,908.8075,NaN,949.3850,NaN,973.3889


In [7]:
# Pathway-descriptor variables (README's "variables to analyse"), kept as full trajectories.
VARS_OF_INTEREST = [
    "Emissions|CO2",
    "Primary Energy",
    "Primary Energy|Fossil",
    "Primary Energy|Non-Biomass Renewables",
    "Primary Energy|Biomass",
    "Primary Energy|Nuclear",
    "Final Energy",
    "Final Energy|Electricity",
    "Carbon Capture|Geological Storage",  # engineered CCS flow (point-source capture + storage)
    "Carbon Removal|Land Use",  # nature-based removal (afforestation, soil carbon, biochar) - distinct lever, weakly correlated with CCS (r~0.23)
    "GDP|PPP",  # billion USD_2010/yr - 1367/1599 scenarios (85%)
    "Population",  # million - 1451/1599 scenarios (91%)
]

missing = set(VARS_OF_INTEREST) - set(data_long["Variable"].unique())
assert not missing, f"Variables not found in data sheet: {missing}"

filtered = data_long[data_long["Variable"].isin(VARS_OF_INTEREST)].copy()
print(f"{len(filtered)} rows across {filtered['Variable'].nunique()} variables")

17857 rows across 12 variables


In [8]:
year_cols = [c for c in filtered.columns if c.isdigit()]

# unstack Variable into columns, keep Model/Scenario as the row key
wide = filtered.set_index(["Model", "Scenario", "Variable"])[year_cols].unstack("Variable")
wide.columns = [f"{var}|{year}" for year, var in wide.columns]
wide = wide.reset_index()

print(wide.shape)
wide.head()

(1599, 230)


,Model,Scenario,Carbon Capture|Geological Storage|2010,Carbon Removal|Land Use|2010,Emissions|CO2|2010,Final Energy|2010,Final Energy|Electricity|2010,GDP|PPP|2010,Population|2010,Primary Energy|2010,Primary Energy|Biomass|2010,Primary Energy|Fossil|2010,Primary Energy|Non-Biomass Renewables|2010,Primary Energy|Nuclear|2010,Carbon Capture|Geological Storage|2015,Carbon Removal|Land Use|2015,Emissions|CO2|2015,Final Energy|2015,Final Energy|Electricity|2015,GDP|PPP|2015,Population|2015,Primary Energy|2015,Primary Energy|Biomass|2015,Primary Energy|Fossil|2015,Primary Energy|Non-Biomass Renewables|2015,Primary Energy|Nuclear|2015,Carbon Capture|Geological Storage|2020,Carbon Removal|Land Use|2020,Emissions|CO2|2020,Final Energy|2020,...,Population|2090,Primary Energy|2090,Primary Energy|Biomass|2090,Primary Energy|Fossil|2090,Primary Energy|Non-Biomass Renewables|2090,Primary Energy|Nuclear|2090,Carbon Capture|Geological Storage|2095,Carbon Removal|Land Use|2095,Emissions|CO2|2095,Final Energy|2095,Final Energy|Electricity|2095,GDP|PPP|2095,Population|2095,Primary Energy|2095,Primary Energy|Biomass|2095,Primary Energy|Fossil|2095,Primary Energy|Non-Biomass Renewables|2095,Primary Energy|Nuclear|2095,Carbon Capture|Geological Storage|2100,Carbon Removal|Land Use|2100,Emissions|CO2|2100,Final Energy|2100,Final Energy|Electricity|2100,GDP|PPP|2100,Population|2100,Primary Energy|2100,Primary Energy|Biomass|2100,Primary Energy|Fossil|2100,Primary Energy|Non-Biomass Renewables|2100,Primary Energy|Nuclear|2100
0,AIM/CGE 2.0,SSP1-19,0.0,NaN,35782.6485,342.2994,64.5446,71310.55217,6879.5896,470.1031,44.9428,400.6963,13.3812,11.0829,0.0,NaN,36508.26915,354.12755,74.22065,89836.671385,7204.40845,491.36250,43.70495,418.17680,17.40440,12.07640,0.0,NaN,37233.8898,365.9557,...,7435.4936,404.5736,86.6641,54.7800,240.2778,22.8518,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5253.5049,NaN,-4474.8741,311.6341,243.8105,606277.1,6881.5846,390.2476,86.7606,46.9562,236.0284,20.5024
1,AIM/CGE 2.0,SSP1-26,0.0,1657.5703,35781.8143,342.3496,64.5452,71311.61510,6879.5896,470.1648,44.9429,400.7582,13.3809,11.0828,0.0,1652.86035,36481.57640,354.17240,74.23840,89838.073450,7204.40845,491.46195,43.70375,418.27880,17.40285,12.07650,0.0,1648.1504,37181.3385,365.9952,...,7435.4936,460.3308,76.8439,171.6095,185.2776,26.5998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9219.4939,3462.2778,17.7377,326.1978,222.4464,608721.3,6881.5846,447.0171,77.9287,164.1606,180.6178,24.3102
2,AIM/CGE 2.0,SSP1-34,0.0,1657.5703,35781.8143,342.3496,64.5452,71311.61510,6879.5896,470.1648,44.9429,400.7582,13.3809,11.0828,0.0,1652.86035,36481.57640,354.17240,74.23840,89838.073450,7204.40845,491.46195,43.70375,418.27880,17.40285,12.07650,0.0,1648.1504,37181.3385,365.9952,...,7435.4936,507.8200,85.0360,245.1817,151.4369,26.1654,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6312.4080,2573.2942,9037.7818,348.4575,212.5913,612448.1,6881.5846,493.8859,87.8550,230.8559,151.7119,23.4631
3,AIM/CGE 2.0,SSP1-45,0.0,1657.5703,35781.8143,342.3496,64.5452,71311.61510,6879.5896,470.1648,44.9429,400.7582,13.3809,11.0828,0.0,1653.76320,36185.22300,352.76025,74.17075,89820.027650,7204.40845,489.25685,43.76595,415.62180,17.78775,12.08135,0.0,1649.9561,36588.6317,363.1709,...,7435.4936,534.6702,73.9360,307.2986,129.4012,24.0344,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.1098,2486.3092,18738.1346,361.3381,207.8181,614156.4,6881.5846,508.0139,76.3293,275.1111,134.0676,22.5059
4,AIM/CGE 2.0,SSP1-Baseline,NaN,1657.6039,35888.7492,342.5024,64.4927,71311.99680,6879.5896,470.7130,44.9441,401.6235,13.2658,10.8796,NaN,1650.54025,37509.54180,356.53640,75.02600,89846.398400,7204.40845,498.09915,43.68170,426.55975,16.49350,11.36420,NaN,1643.4766,39130.3344,370.5704,...,7435.4936,597.0849,71.7243,394.9441,118.5924,11.8241,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1605.7374,29879.8054,387.1160,213.5488,616112.2,6881.5846,576.5366,71.7565,374.7253,119.5191,10.5357


## 3. Merge meta + pivoted pathway data

Left join on `meta` (the master Model-Scenario list: 1599 rows) so scenarios missing one of the
pathway variables still appear, just with NaNs for that variable.

In [9]:
merged = meta.merge(wide, on=["Model", "Scenario"], how="left", validate="one_to_one")
print(merged.shape)
merged.head()

(1599, 299)


,Model,Scenario,Climate Assessment|Exceedance Probability 1.5°C [MAGICCv7.5.3],Climate Assessment|Exceedance Probability 2.0°C [MAGICCv7.5.3],Climate Assessment|Exceedance Probability 2.5°C [MAGICCv7.5.3],Climate Assessment|Exceedance Probability 3.0°C [MAGICCv7.5.3],Climate Assessment|Peak Warming|33rd Percentile [MAGICCv7.5.3],Climate Assessment|Peak Warming|67th Percentile [MAGICCv7.5.3],Climate Assessment|Peak Warming|Median [MAGICCv7.5.3],Climate Assessment|Warming in 2100|33rd Percentile [MAGICCv7.5.3],Climate Assessment|Warming in 2100|67th Percentile [MAGICCv7.5.3],Climate Assessment|Warming in 2100|Median [MAGICCv7.5.3],Climate Assessment|Year of Peak Warming|33rd Percentile [MAGICCv7.5.3],Climate Assessment|Year of Peak Warming|67th Percentile [MAGICCv7.5.3],Climate Assessment|Year of Peak Warming|Median [MAGICCv7.5.3],Climate Category|AR6 [ID],Climate Category|AR6 [Name],Climate Category|SCI 2025 [Tier III],Climate Category|SCI 2025 [Tier II],Climate Category|SCI 2025 [Tier I],Data Source (DOI),"Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]","Emissions Diagnostics|Cumulative CO2 [2020-2100, Gt CO2]","Emissions Diagnostics|Cumulative Kyoto Gases [2020-2100, Gt CO2e]","Emissions Diagnostics|Cumulative Net-Negative CO2 [2020-2100, Gt CO2]",Emissions Diagnostics|Year of Net Zero|CO2,Emissions Diagnostics|Year of Net Zero|Kyoto Gases,Feasibility Concern|Carbon Capture|World|2030,Feasibility Concern|Carbon Capture|World|2035,Feasibility Concern|Carbon Capture|World|2040,...,Population|2090,Primary Energy|2090,Primary Energy|Biomass|2090,Primary Energy|Fossil|2090,Primary Energy|Non-Biomass Renewables|2090,Primary Energy|Nuclear|2090,Carbon Capture|Geological Storage|2095,Carbon Removal|Land Use|2095,Emissions|CO2|2095,Final Energy|2095,Final Energy|Electricity|2095,GDP|PPP|2095,Population|2095,Primary Energy|2095,Primary Energy|Biomass|2095,Primary Energy|Fossil|2095,Primary Energy|Non-Biomass Renewables|2095,Primary Energy|Nuclear|2095,Carbon Capture|Geological Storage|2100,Carbon Removal|Land Use|2100,Emissions|CO2|2100,Final Energy|2100,Final Energy|Electricity|2100,GDP|PPP|2100,Population|2100,Primary Energy|2100,Primary Energy|Biomass|2100,Primary Energy|Fossil|2100,Primary Energy|Non-Biomass Renewables|2100,Primary Energy|Nuclear|2100
0,AIM/CGE 2.0,SSP1-19,53.166667,6.500000,0.166667,0.000000,1.411097,1.628765,1.518461,1.001548,1.253812,1.132453,2037.0,2039.0,2038.0,C1,C1: Below 1.5°C with no or limited overshoot,GW1a,GW1,GW1,NaN,348.340310,324.319294,665.101646,-166.408314,2055.0,2066.0,high,medium,ok,...,7435.4936,404.5736,86.6641,54.7800,240.2778,22.8518,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5253.5049,NaN,-4474.8741,311.6341,243.8105,606277.1,6881.5846,390.2476,86.7606,46.9562,236.0284,20.5024
1,AIM/CGE 2.0,SSP1-26,74.500000,21.333333,5.666667,0.666667,1.561912,1.844314,1.702569,1.443959,1.789941,1.609574,2049.0,2069.0,2059.0,C3,C3: Likely below 2°C,GW2-IIIb,GW2b,GW2,NaN,455.449049,988.478191,1501.479227,NaN,NaN,NaN,medium,ok,ok,...,7435.4936,460.3308,76.8439,171.6095,185.2776,26.5998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9219.4939,3462.2778,17.7377,326.1978,222.4464,608721.3,6881.5846,447.0171,77.9287,164.1606,180.6178,24.3102
2,AIM/CGE 2.0,SSP1-34,93.500000,57.666667,22.166667,8.833333,1.908544,2.320240,2.098823,1.900225,2.320240,2.090244,2094.0,2100.0,2100.0,C5,C5: Below 2.5°C,GW5b,GW5,GW5,NaN,159.484812,1808.796903,2467.636562,NaN,NaN,NaN,medium,ok,ok,...,7435.4936,507.8200,85.0360,245.1817,151.4369,26.1654,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6312.4080,2573.2942,9037.7818,348.4575,212.5913,612448.1,6881.5846,493.8859,87.8550,230.8559,151.7119,23.4631
3,AIM/CGE 2.0,SSP1-45,99.833333,90.000000,61.166667,28.000000,2.419310,2.918826,2.666972,2.418018,2.918826,2.664423,2100.0,2100.0,2100.0,C6,C6: Below 3.0°C,GW6,GW6,GW6,NaN,0.226104,2500.585275,3485.276061,NaN,NaN,NaN,medium,ok,ok,...,7435.4936,534.6702,73.9360,307.2986,129.4012,24.0344,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

## 3b. Derive GDP per capita

`GDP|PPP` is in billion USD_2010/yr, `Population` in million people, so
`GDP|PPP / Population * 1000` gives GDP per capita in thousand USD_2010/cap/yr. Derived for
every year both are reported (1304/1599 scenarios, 82%, have both).

In [10]:
gdp_cols = sorted(
    [c for c in merged.columns if c.startswith("GDP|PPP|")],
    key=lambda c: int(c.split("|")[-1]),
)
gdp_years = [c.split("|")[-1] for c in gdp_cols]

for year in gdp_years:
    merged[f"GDP per capita|PPP|{year}"] = (
        merged[f"GDP|PPP|{year}"] / merged[f"Population|{year}"] * 1000
    )

n_with_gdp_pc = merged["GDP per capita|PPP|2050"].notna().sum()
print(f"GDP per capita derived for {len(gdp_years)} years; non-null at 2050: {n_with_gdp_pc}")

GDP per capita derived for 19 years; non-null at 2050: 1304


## 4. Build the net-zero-by-2070 outcome labels

- `success_nz2070`: reaches net-zero CO2 at or before 2070 (NaN year = never reaches net zero within
  the model horizon -> treated as failure)
- `failure_nz2070`: complement of success
- `sustained_success_nz2070`: reaches net zero by 2070 *and* stays at or below zero every reported year
  through 2100 (no rebound)

In [11]:
NZ_COL = "Emissions Diagnostics|Year of Net Zero|CO2"

merged["success_nz2070"] = merged[NZ_COL].le(2070)
merged["failure_nz2070"] = ~merged["success_nz2070"]

co2_cols = sorted(
    [c for c in merged.columns if c.startswith("Emissions|CO2|")],
    key=lambda c: int(c.split("|")[-1]),
)
co2_years = np.array([int(c.split("|")[-1]) for c in co2_cols])


def stays_at_or_below_zero_after_nz(row):
    nz_year = row[NZ_COL]
    if pd.isna(nz_year):
        return False
    after_mask = co2_years >= nz_year
    vals = row[np.array(co2_cols)[after_mask]]
    if vals.isna().any():
        return False  # can't confirm "sustained" if the trajectory has gaps after net zero
    return bool((vals <= 0).all())


merged["sustained_success_nz2070"] = merged["success_nz2070"] & merged.apply(
    stays_at_or_below_zero_after_nz, axis=1
)

merged[["success_nz2070", "failure_nz2070", "sustained_success_nz2070"]].sum()

success_nz2070               497
failure_nz2070              1102
sustained_success_nz2070     135
dtype: int64

## 4b. Refine the desired space: net zero *and* low CCS reliance

`success_nz2070` alone doesn't distinguish pathways that reach net zero through early mitigation
from pathways that lean heavily on carbon capture and storage to get there. We add
`Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]` (94% coverage in `meta`, no extra
data needed) as a second criterion: a scenario is in the refined "desired" region only if it
reaches net zero by 2070 *and* its cumulative CCS use is at or below the ensemble median.

The median is computed over the full ensemble (not just successes), so the threshold is a fixed,
externally interpretable cutoff rather than a self-referential one. Scenarios with no reported
cumulative CCS (6% of the ensemble) cannot be confirmed "low reliance" and are excluded from the
desired region rather than silently counted as low.

In [12]:
CCS_COL = "Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]"

ccs_median = 1000
merged["low_ccs_reliance"] = merged[CCS_COL] <= ccs_median
merged["desired_success"] = merged["success_nz2070"] & merged["low_ccs_reliance"]

print(f"Median cumulative CCS (2020-2100): {ccs_median:.1f} Gt CO2")
print(f"Missing cumulative CCS: {merged[CCS_COL].isna().sum()} scenarios")
print()
print("Cross-tab: success_nz2070 x low_ccs_reliance")
print(pd.crosstab(merged["success_nz2070"], merged["low_ccs_reliance"], dropna=False, margins=True))
print()
print(f"desired_success (NZ<=2070 AND low CCS reliance): {merged['desired_success'].sum()} scenarios")

Median cumulative CCS (2020-2100): 1000.0 Gt CO2
Missing cumulative CCS: 90 scenarios

Cross-tab: success_nz2070 x low_ccs_reliance
low_ccs_reliance  False  True   All
success_nz2070                     
False               147   955  1102
True                125   372   497
All                 272  1327  1599

desired_success (NZ<=2070 AND low CCS reliance): 372 scenarios


## 4c. Ramp-up pace: progressive build-out vs late intensive ramp-up

Two scenarios can reach the same 2050/2060 deployment level for a technology via very different paces:
steady growth from early on, or a slow start followed by a steep late ramp. We capture this by
comparing growth over two consecutive periods for each technology:

- early period: 2020 -> 2040
- late period: 2040 -> 2060

`late_growth_share = late_growth / (early_growth + late_growth)` is the fraction of the total
2020-2060 growth that happens in the second half - bounded near [0, 1] for monotonically growing
technologies. We compute this for the technologies most relevant to "ramp-up reliance": non-biomass
renewables (the main renewables proxy used throughout), final energy electrification, nuclear, and
CCS. A scenario with `late_growth_share` well above 0.5 is leaning on a late, more intensive ramp-up
for that technology rather than a progressive build-out.

Caveat: the ratio is undefined (set to NaN) when total growth over 2020-2060 is ~0 (nothing to take a
share of), and can fall outside [0, 1] if the technology shrinks in one period and grows in the other.

In [13]:
RAMP_VARS = [
    "Primary Energy|Non-Biomass Renewables",
    "Final Energy|Electricity",
    "Primary Energy|Nuclear",
    "Carbon Capture|Geological Storage",
]
EARLY_START, MID, LATE_END = 2020, 2040, 2060

for var in RAMP_VARS:
    v_early_start = merged[f"{var}|{EARLY_START}"]
    v_mid = merged[f"{var}|{MID}"]
    v_late_end = merged[f"{var}|{LATE_END}"]

    early_growth = v_mid - v_early_start
    late_growth = v_late_end - v_mid
    total_growth = early_growth + late_growth

    late_share = late_growth / total_growth
    late_share = late_share.replace([np.inf, -np.inf], np.nan)
    late_share[total_growth.abs() < 1.0] = np.nan  # nothing meaningful to take a share of

    merged[f"{var}|growth_{EARLY_START}_{MID}"] = early_growth
    merged[f"{var}|growth_{MID}_{LATE_END}"] = late_growth
    merged[f"{var}|late_growth_share"] = late_share
    merged[f"{var}|ramp_profile"] = pd.cut(
        late_share, bins=[-np.inf, 0.4, 0.6, np.inf],
        labels=["front-loaded", "balanced", "back-loaded"],
    )

    print(f"{var}: late_growth_share non-null = {late_share.notna().sum()}, "
          f"median = {late_share.median():.2f}")
    print(merged[f"{var}|ramp_profile"].value_counts(dropna=False).to_dict())

Primary Energy|Non-Biomass Renewables: late_growth_share non-null = 1442, median = 0.57
{'balanced': 862, 'back-loaded': 429, nan: 157, 'front-loaded': 151}
Final Energy|Electricity: late_growth_share non-null = 1538, median = 0.59
{'balanced': 774, 'back-loaded': 691, 'front-loaded': 73, nan: 61}
Primary Energy|Nuclear: late_growth_share non-null = 1446, median = 0.59
{'back-loaded': 692, 'balanced': 437, 'front-loaded': 317, nan: 153}
Carbon Capture|Geological Storage: late_growth_share non-null = 1358, median = 0.70
{'back-loaded': 926, 'balanced': 280, nan: 241, 'front-loaded': 152}


## 5. Sanity checks

In [14]:
print("Rows:", len(merged))
print("Success / failure counts:")
print(merged["success_nz2070"].value_counts())
print()
print("Missingness on key pathway variables (2050 snapshot):")
for v in VARS_OF_INTEREST:
    col = f"{v}|2050"
    if col in merged.columns:
        print(f"  {col}: {merged[col].isna().mean():.1%} missing")

Rows: 1599
Success / failure counts:
success_nz2070
False    1102
True      497
Name: count, dtype: int64

Missingness on key pathway variables (2050 snapshot):
  Emissions|CO2|2050: 0.9% missing
  Primary Energy|2050: 1.1% missing
  Primary Energy|Fossil|2050: 1.5% missing
  Primary Energy|Non-Biomass Renewables|2050: 8.7% missing
  Primary Energy|Biomass|2050: 1.3% missing
  Primary Energy|Nuclear|2050: 1.0% missing
  Final Energy|2050: 1.4% missing
  Final Energy|Electricity|2050: 2.6% missing
  Carbon Capture|Geological Storage|2050: 5.4% missing
  Carbon Removal|Land Use|2050: 35.9% missing
  GDP|PPP|2050: 14.5% missing
  Population|2050: 9.3% missing


In [15]:
merged.groupby("success_nz2070")[["Emissions|CO2|2030", "Primary Energy|Fossil|2050"]].median()

,Emissions|CO2|2030,Primary Energy|Fossil|2050
success_nz2070,,
False,37987.765954,427.186484
True,28327.499100,209.639716


## 6. Save

In [16]:
merged.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {merged.shape} to {OUTPUT_CSV}")

Saved (1599, 339) to data_for_scenariodiscovery_full.csv
